# 基于机器学习的命名实体识别方法

## 1. 学习目标

完成本课程后，你应该能够：

- 理解将NER看作序列标注问题
- 掌握BIO标注方案
- 了解条件随机场（CRF）的基本原理
- 能够使用CRF模型进行实体识别
- 理解特征工程在NER中的重要性

## 2. 从规则到机器学习

### 2.1 规则方法的局限性

在上一节课中，我们学习了基于规则的方法（词典匹配和正则表达式）。

**这些方法的问题：**

❌ 无法识别词典中没有的新实体
```
"孙颖莎获得冠军" → 如果"孙颖莎"不在词典中，无法识别
```

❌ 无法处理歧义
```
"苹果公司" → 是机构名
"买了苹果" → 不是实体
同一个词"苹果"，规则方法难以区分
```

❌ 需要大量人工编写规则
```
每个领域、每种实体类型都需要单独编写规则
维护成本高
```

### 2.2 机器学习方法的优势

**基于机器学习的方法**：让计算机从大量已标注的数据中自动学习规律

**优势：**

✅ 可以识别未登录词（新词）
```
通过学习人名的特征模式（如常用姓氏、名字结构），
即使"孙颖莎"不在训练数据中，也可能识别出来
```

✅ 可以利用上下文消歧
```
"苹果公司发布新品" → 根据"公司"、"发布"等上下文，判断是机构名
"买了一斤苹果" → 根据"一斤"等上下文，判断不是实体
```

✅ 不需要人工编写大量规则
```
只需要准备标注数据，模型自动学习规律
```

## 3. 序列标注问题

### 3.1 什么是序列标注？

**序列标注（Sequence Labeling）**：给序列中的每个元素分配一个标签

**在NER中：**
- **序列**：一个句子中的字或词
- **标注**：给每个字或词标上实体类别标签

**示例：**

```
输入序列：马  云  在  杭  州  创  立  阿  里  巴  巴
输出标签：PER PER O  LOC LOC O  O  ORG ORG ORG ORG

说明：
PER = 人名（Person）
LOC = 地名（Location）
ORG = 机构名（Organization）
O = 非实体（Outside）
```

**类比理解：**

就像给句子中的每个字涂颜色：
- 人名涂红色
- 地名涂蓝色
- 机构名涂绿色
- 其他不涂色

### 3.2 BIO标注方案

**问题**：如何区分一个实体的开始和中间部分？

```
"马云" 和 "阿里巴巴" 都是实体
如果都标注为 PER PER 和 ORG ORG ORG ORG
无法区分 "马" "云" 是一个实体还是两个实体
```

**解决方案：BIO标注方案**

**BIO的含义：**
- **B（Begin）**：实体的开始
- **I（Inside）**：实体的中间或结尾
- **O（Outside）**：非实体

**标注示例：**

| 字 | 标签 | 说明 |
|---|------|------|
| 马 | B-PER | 人名的开始 |
| 云 | I-PER | 人名的中间 |
| 在 | O | 非实体 |
| 杭 | B-LOC | 地名的开始 |
| 州 | I-LOC | 地名的中间 |
| 创 | O | 非实体 |
| 立 | O | 非实体 |
| 阿 | B-ORG | 机构名的开始 |
| 里 | I-ORG | 机构名的中间 |
| 巴 | I-ORG | 机构名的中间 |
| 巴 | I-ORG | 机构名的结尾 |

**好处：**
- 明确标识实体的边界
- 可以区分连续的多个实体

### 3.3 BIO标注的更多示例

**示例1：连续的两个人名**

```
文本："张三和李四是朋友"

字：  张  三  和  李  四  是  朋  友
标签：B-PER I-PER O B-PER I-PER O O O

识别结果：
- 实体1：张三（PER）
- 实体2：李四（PER）
```

**示例2：嵌套在一起的实体**

```
文本："去北京大学参观"

字：  去  北  京  大  学  参  观
标签：O  B-ORG I-ORG I-ORG I-ORG O O

识别结果：
- 实体：北京大学（ORG）

注意：虽然"北京"本身是地名，但在这里作为"北京大学"的一部分
```

## 4. 条件随机场（CRF）简介

### 4.1 什么是CRF？

**条件随机场（Conditional Random Field，CRF）**：一种用于序列标注的机器学习模型

**简单理解：**

CRF是一个"聪明的打标签工具"，它在给每个字打标签时会考虑：

**1. 当前字的特征**
- 这个字是什么？
- 是汉字还是数字？
- 是否在词典中？

**2. 上下文特征**
- 前面的字是什么？
- 后面的字是什么？
- 前面的词是什么？

**3. 标签之间的关系**
- B-PER 后面通常跟 I-PER（而不是 B-LOC）
- I-PER 前面通常是 B-PER 或 I-PER
- O 后面可以是任何标签

**类比：**

就像医生看病：
- 不只看一个症状（当前字）
- 要结合病史（前面的字）、家族史（前面的标签）
- 还要考虑症状之间的关联（标签之间的转移规律）

### 4.2 CRF的优势

**与简单分类器相比：**

❌ **简单分类器**：单独看每个字
```
"云" → 可能是人名的一部分，也可能是"云朵"的"云"
单独看"云"，很难判断
```

✅ **CRF**：同时看整个句子
```
"马云在杭州" → 看到"马""云"在一起，前面没有量词，后面是"在"
→ 综合判断"马云"是人名
```

**CRF保证标签序列合理：**

❌ 不合理的标签序列
```
B-PER I-LOC  → I-LOC不能跟在B-PER后面
I-PER B-PER  → 两个B标签不能连续（中间应该有O）
```

✅ CRF会避免这些不合理的情况
```
B-PER I-PER O  → 合理
B-PER I-PER I-PER → 合理
```

### 4.3 什么是特征？

在机器学习中，**特征**就是我们提供给模型的"线索"或"证据"，帮助模型做出判断。

**生活中的类比：**

就像你判断一个人是不是学生，你会看：
- 年龄（特征1）
- 穿着（特征2）
- 背着书包（特征3）
- 拿着课本（特征4）

这些都是"特征"，帮助你做出判断。

**在NER中的特征：**

判断"云"是不是人名的一部分，我们可以看：
- 当前字是什么（"云"）
- 前面的字是什么（"马"）
- 后面的字是什么（"在"）
- "马"是不是常见姓氏（是）
- "在"是不是介词（是）

这些都是特征，CRF会根据这些特征来判断。

### 4.4 特征的具体示例

让我们通过一个例子来理解特征。

In [ ]:
# 示例：为"马云在杭州"中的每个字提取简单特征

sentence = "马云在杭州"
chars = list(sentence)

print("我们要为每个字提取特征：\n")

# 为第2个字"云"（索引为1）提取特征
i = 1
features = {
    '当前字': chars[i],           # 云
    '前一个字': chars[i-1],        # 马
    '后一个字': chars[i+1],        # 在
}

print(f"第{i+1}个字 '{chars[i]}' 的特征：")
for key, value in features.items():
    print(f"  {key}: {value}")

**输出说明：**

这些特征告诉CRF：
- 当前字是"云"
- 它前面是"马"（常见姓氏）
- 它后面是"在"（介词）

CRF会从训练数据中学到：当出现"姓氏 + 某字 + 介词"这种模式时，前两个字很可能是人名。

### 4.5 常用特征类型

在实际应用中，我们会提取更多类型的特征：

**1. 字符特征**
- 当前字是什么
- 前一个字、后一个字
- 前两个字、后两个字

**2. 词特征**
- 当前词是什么
- 前一个词、后一个词

**3. 词性特征**
- 当前词的词性（名词/动词/形容词等）

**4. 词典特征**
- 是否在人名词典中
- 是否在地名词典中
- 是否在机构名词典中

**5. 字符类型特征**
- 是否是数字
- 是否是字母
- 是否是标点符号

**特征越丰富，模型效果越好！**

## 5. 使用CRF进行实体识别实践

### 5.1 安装和导入库

我们将使用 `sklearn-crfsuite` 库来实现CRF模型。

**需要用到的Python知识：**

**1. 字典（dict）**：存储特征
```python
features = {'char': '云', 'prev_char': '马'}
```

**2. 列表推导式**：快速创建列表
```python
[char for char in '马云']  # ['马', '云']
```

**3. enumerate() 函数**：遍历时获取索引
```python
for i, char in enumerate(['马', '云']):
    print(i, char)  # 0 马, 1 云
```

In [ ]:
# 安装必要的库
!pip install sklearn-crfsuite -q

导入需要的库

In [ ]:
# 导入需要的库
import jieba
import jieba.posseg as pseg  # 词性标注
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_f1_score, flat_classification_report

print("✅ 库导入成功！")

### 5.2 理解训练数据格式

在训练CRF模型之前，我们需要准备标注好的数据。

**数据格式：**
- 每个句子是一个列表
- 列表中的每个元素是一个元组：`(字, 标签)`

**示例：**

In [ ]:
# 一个简单的示例句子："马云在杭州"
example_sentence = [
    ('马', 'B-PER'),  # 马 - 人名的开始
    ('云', 'I-PER'),  # 云 - 人名的中间
    ('在', 'O'),      # 在 - 非实体
    ('杭', 'B-LOC'),  # 杭 - 地名的开始
    ('州', 'I-LOC'),  # 州 - 地名的中间
]

print("示例句子的数据格式：")
for char, label in example_sentence:
    print(f"  字: {char}  →  标签: {label}")

**说明：**

- `('马', 'B-PER')` 表示："马"这个字的标签是 B-PER（人名的开始）
- `('在', 'O')` 表示："在"这个字的标签是 O（不是实体）

这种格式让计算机知道：哪些字是实体，哪些不是。

### 5.3 准备训练数据

现在我们准备一些训练句子。为了教学演示，我们准备少量但有代表性的数据。

**训练数据的作用：**
- 让CRF学习：什么样的字序列是人名、地名、机构名
- 学习不同标签之间的转移规律

In [ ]:
# 准备训练数据（8个句子）
train_data = [
    # 句子1：马云在杭州创立阿里巴巴
    [('马', 'B-PER'), ('云', 'I-PER'), ('在', 'O'), ('杭', 'B-LOC'), ('州', 'I-LOC'), 
     ('创', 'O'), ('立', 'O'), ('阿', 'B-ORG'), ('里', 'I-ORG'), ('巴', 'I-ORG'), ('巴', 'I-ORG')],
    
    # 句子2：马化腾在深圳创办了腾讯
    [('马', 'B-PER'), ('化', 'I-PER'), ('腾', 'I-PER'), ('在', 'O'), ('深', 'B-LOC'), ('圳', 'I-LOC'),
     ('创', 'O'), ('办', 'O'), ('了', 'O'), ('腾', 'B-ORG'), ('讯', 'I-ORG')],
    
    # 句子3：李彦宏把百度总部设在北京
    [('李', 'B-PER'), ('彦', 'I-PER'), ('宏', 'I-PER'), ('把', 'O'), ('百', 'B-ORG'), ('度', 'I-ORG'),
     ('总', 'O'), ('部', 'O'), ('设', 'O'), ('在', 'O'), ('北', 'B-LOC'), ('京', 'I-LOC')],
    
    # 句子4：雷军创办小米
    [('雷', 'B-PER'), ('军', 'I-PER'), ('创', 'O'), ('办', 'O'), ('小', 'B-ORG'), ('米', 'I-ORG')],
]

print(f"训练数据：{len(train_data)} 个句子")
print(f"\n第1个句子：{train_data[0]}")

In [ ]:
# 再准备4个训练句子
more_train_data = [
    # 句子5：刘强东在北京创立京东
    [('刘', 'B-PER'), ('强', 'I-PER'), ('东', 'I-PER'), ('在', 'O'), ('北', 'B-LOC'), ('京', 'I-LOC'),
     ('创', 'O'), ('立', 'O'), ('京', 'B-ORG'), ('东', 'I-ORG')],
    
    # 句子6：华为总部位于深圳
    [('华', 'B-ORG'), ('为', 'I-ORG'), ('总', 'O'), ('部', 'O'), ('位', 'O'), ('于', 'O'),
     ('深', 'B-LOC'), ('圳', 'I-LOC')],
    
    # 句子7：张伟在上海工作
    [('张', 'B-PER'), ('伟', 'I-PER'), ('在', 'O'), ('上', 'B-LOC'), ('海', 'I-LOC'), 
     ('工', 'O'), ('作', 'O')],
    
    # 句子8：小米公司在北京
    [('小', 'B-ORG'), ('米', 'I-ORG'), ('公', 'O'), ('司', 'O'), ('在', 'O'), 
     ('北', 'B-LOC'), ('京', 'I-LOC')],
]

# 合并所有训练数据
train_data.extend(more_train_data)

print(f"现在共有 {len(train_data)} 个训练句子")

### 5.4 准备测试数据

测试数据用于评估模型的效果，看看它能不能正确识别从未见过的句子。

In [ ]:
# 准备测试数据（2个句子）
test_data = [
    # 句子1：王明在广州创业
    [('王', 'B-PER'), ('明', 'I-PER'), ('在', 'O'), ('广', 'B-LOC'), ('州', 'I-LOC'), 
     ('创', 'O'), ('业', 'O')],
    
    # 句子2：字节跳动在北京
    [('字', 'B-ORG'), ('节', 'I-ORG'), ('跳', 'I-ORG'), ('动', 'I-ORG'), ('在', 'O'),
     ('北', 'B-LOC'), ('京', 'I-LOC')],
]

print(f"测试数据：{len(test_data)} 个句子")
print(f"\n第1个测试句子：{test_data[0]}")

**说明：**

- 训练数据：8个句子，用来教CRF学习规律
- 测试数据：2个句子，用来检验CRF学得怎么样

注意："王明"和"字节跳动"可能不在训练数据中，我们要看CRF能否通过学到的规律识别它们。

### 5.5 特征提取 - 第1步：基本特征

现在我们要写一个函数，为每个字提取特征。

我们先从最简单的特征开始：**当前字是什么**。

In [ ]:
# 第1版：只提取当前字的信息
def char2features_v1(sent, i):
    """
    提取第i个字符的基本特征
    
    参数:
        sent: 句子，格式为 [(字, 标签), (字, 标签), ...]
        i: 字符在句子中的位置（索引，从0开始）
    """
    char = sent[i][0]  # 获取第i个字
    
    features = {
        'char': char,  # 当前字是什么
    }
    
    return features

# 测试
example = train_data[0]  # "马云在杭州创立阿里巴巴"
print("测试句子：", example[:5])  # 只显示前5个字
print("\n第2个字（索引1）的特征：")
print(char2features_v1(example, 1))

**说明：**

- `sent[i][0]` 获取第i个元组的第一个元素（字符）
- 例如：`sent[1]` 是 `('云', 'I-PER')`，`sent[1][0]` 就是 `'云'`

但只有当前字还不够，我们需要加入更多特征！

### 5.6 特征提取 - 第2步：加入上下文

现在我们加入前一个字和后一个字的信息。

In [ ]:
# 第2版：加入前后字的信息
def char2features_v2(sent, i):
    """
    提取第i个字符的特征（包含上下文）
    """
    char = sent[i][0]
    
    features = {
        'char': char,  # 当前字
    }
    
    # 如果不是第一个字，加入前一个字的信息
    if i > 0:
        prev_char = sent[i-1][0]
        features['prev_char'] = prev_char  # 前一个字
    else:
        features['BOS'] = True  # BOS = Beginning Of Sentence（句子开始）
    
    # 如果不是最后一个字，加入后一个字的信息
    if i < len(sent) - 1:
        next_char = sent[i+1][0]
        features['next_char'] = next_char  # 后一个字
    else:
        features['EOS'] = True  # EOS = End Of Sentence（句子结束）
    
    return features

# 测试
print("第2个字（索引1）'云'的特征：")
print(char2features_v2(example, 1))
print("\n第1个字（索引0）'马'的特征：")
print(char2features_v2(example, 0))

**说明：**

- 对于"云"：有前一个字"马"和后一个字"在"
- 对于"马"：是句子开头，所以标记 `BOS=True`，没有前一个字

这些上下文信息非常重要，帮助CRF判断实体边界。

### 5.7 特征提取 - 第3步：完整版本

最后，我们加入更多特征：字符类型、前后两个字等。

In [ ]:
# 完整版：提取丰富的特征
def char2features(sent, i):
    """
    提取第i个字符的完整特征
    """
    char = sent[i][0]
    
    # 基本特征
    features = {
        'char': char,                    # 当前字
        'is_digit': char.isdigit(),      # 是否是数字
        'is_alpha': char.isalpha(),      # 是否是字母
    }
    
    # 前一个字的特征
    if i > 0:
        prev_char = sent[i-1][0]
        features['prev_char'] = prev_char
        features['prev_is_digit'] = prev_char.isdigit()
    else:
        features['BOS'] = True
    
    # 后一个字的特征
    if i < len(sent) - 1:
        next_char = sent[i+1][0]
        features['next_char'] = next_char
        features['next_is_digit'] = next_char.isdigit()
    else:
        features['EOS'] = True
    
    # 前两个字
    if i > 1:
        features['prev2_char'] = sent[i-2][0]
    
    # 后两个字
    if i < len(sent) - 2:
        features['next2_char'] = sent[i+2][0]
    
    return features

# 测试
print("完整版特征 - 第2个字（索引1）'云'：")
features = char2features(example, 1)
for key, value in features.items():
    print(f"  {key}: {value}")

**特征说明：**

- `char`: 当前字是"云"
- `is_digit`: False（不是数字）
- `is_alpha`: True（是字母/汉字）
- `prev_char`: 前一个字是"马"
- `next_char`: 后一个字是"在"
- `next2_char`: 后两个字是"杭"

这些特征帮助CRF全面了解"云"这个字的上下文环境。

### 5.8 辅助函数：处理整个句子

我们需要一些辅助函数来处理整个句子的特征和标签。

In [ ]:
# 函数1：为整个句子提取特征
def sent2features(sent):
    """
    为句子中的每个字提取特征
    
    返回：特征列表，每个元素是一个字的特征字典
    """
    return [char2features(sent, i) for i in range(len(sent))]

# 测试
print("示例句子的前3个字：", example[:3])
features = sent2features(example)
print(f"\n提取了 {len(features)} 个字的特征")
print(f"第1个字的特征有 {len(features[0])} 个特征项")

In [ ]:
# 函数2：提取句子的标签序列
def sent2labels(sent):
    """
    提取句子的标签序列
    
    返回：标签列表
    """
    return [label for char, label in sent]

# 测试
labels = sent2labels(example)
print("标签序列：", labels)

In [ ]:
# 函数3：提取句子的字符序列
def sent2chars(sent):
    """
    提取句子的字符序列
    """
    return [char for char, label in sent]

# 测试
chars = sent2chars(example)
print("字符序列：", chars)

**函数总结：**

- `sent2features(sent)` - 提取所有字的特征
- `sent2labels(sent)` - 提取所有字的标签
- `sent2chars(sent)` - 提取所有字符

这些函数帮助我们快速处理数据。

### 5.9 准备训练和测试数据

现在我们为所有句子提取特征和标签。

In [ ]:
# 为训练数据提取特征和标签
X_train = [sent2features(s) for s in train_data]
y_train = [sent2labels(s) for s in train_data]

print("训练数据准备完成：")
print(f"  句子数：{len(X_train)}")
print(f"  第1个句子有 {len(X_train[0])} 个字")
print(f"  第1个句子的标签：{y_train[0]}")

In [ ]:
# 为测试数据提取特征和标签
X_test = [sent2features(s) for s in test_data]
y_test = [sent2labels(s) for s in test_data]

print("测试数据准备完成：")
print(f"  句子数：{len(X_test)}")
print(f"  第1个句子有 {len(X_test[0])} 个字")
print(f"  第1个句子的标签：{y_test[0]}")

**数据格式说明：**

- `X_train` 和 `X_test`：特征数据，每个句子是一个特征列表
- `y_train` 和 `y_test`：标签数据，每个句子是一个标签列表

现在数据已经准备好，可以训练模型了！

### 5.10 训练CRF模型

现在我们创建并训练CRF模型。

**CRF主要参数说明：**

- `algorithm='lbfgs'`：优化算法（一种数学优化方法）
- `c1=0.1`：L1正则化系数（防止模型过度拟合）
- `c2=0.1`：L2正则化系数（防止模型过度拟合）
- `max_iterations=100`：最多训练100次
- `all_possible_transitions=True`：考虑所有可能的标签转换

**什么是过度拟合？**

就像学生死记硬背考题答案，遇到新题就不会了。c1和c2参数帮助模型学习通用规律而不是死记硬背。

In [ ]:
# 创建CRF模型
crf = CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("CRF模型创建成功！")
print("\n模型参数：")
print(f"  优化算法: {crf.algorithm}")
print(f"  最大迭代次数: {crf.max_iterations}")

In [ ]:
# 训练模型
print("开始训练CRF模型...")
print("（这可能需要几秒钟时间）\n")

crf.fit(X_train, y_train)

print("✅ 模型训练完成！")
print(f"\n模型学习了 {len(crf.classes_)} 种标签：")
print(crf.classes_)

**训练过程说明：**

在训练过程中，CRF学习了：
- 每种特征对不同标签的影响（例如："马"后面跟"云"，很可能是人名）
- 标签之间的转移规律（例如：B-PER后面通常跟I-PER）

训练完成后，模型就可以用来识别新句子中的实体了！

### 5.11 模型评估

训练完成后，我们要评估模型在测试集上的效果。

**评估指标说明：**

我们使用 **F1分数** 来评估模型效果。F1分数综合考虑了：

1. **准确率（Precision）**：模型识别出的实体中，有多少是正确的
   - 例如：识别出10个人名，其中8个是对的，准确率 = 8/10 = 80%

2. **召回率（Recall）**：所有真实的实体中，模型识别出了多少
   - 例如：句子中有10个人名，模型识别出了8个，召回率 = 8/10 = 80%

3. **F1分数**：准确率和召回率的综合评分
   - F1 = 2 × (准确率 × 召回率) / (准确率 + 召回率)
   - F1分数越高越好，最高是100%

In [ ]:
# 在测试集上预测
y_pred = crf.predict(X_test)

print("预测完成！")
print(f"\n第1个测试句子的预测结果：")
print(f"真实标签：{y_test[0]}")
print(f"预测标签：{y_pred[0]}")

In [ ]:
# 计算F1分数
# 只关注实体标签（B-PER, I-PER, B-LOC等），不计算O标签
labels = list(crf.classes_)
labels.remove('O')  # 移除'O'标签

f1 = flat_f1_score(y_test, y_pred, average='weighted', labels=labels)

print(f"\nF1分数: {f1:.2%}")
print("\n说明：F1分数越高越好，100%表示完全正确")

In [ ]:
# 详细的分类报告
print("\n详细评估报告：")
print("="*60)
print(flat_classification_report(y_test, y_pred, labels=labels, digits=3))
print("\n报告说明：")
print("  precision（准确率）：识别出的实体中正确的比例")
print("  recall（召回率）：真实实体中被识别出的比例")
print("  f1-score（F1分数）：准确率和召回率的综合")
print("  support（支持数）：测试数据中该标签的实际数量")

**如何理解评估报告？**

看这个例子：

```
       precision  recall  f1-score  support
B-PER     0.50     1.00      0.67        1
```

- **support = 1**：测试数据中有1个B-PER标签
- **recall = 1.00**：这1个B-PER都被识别出来了（100%）
- **precision = 0.50**：但模型识别出的B-PER中只有一半是对的（50%）
- **f1-score = 0.67**：综合评分67%

**注意**：我们的训练数据很少（只有8个句子），所以效果有限。实际应用中需要更多训练数据。

### 5.12 测试新句子 - 第1步：理解识别流程

现在我们要用训练好的模型来识别新句子中的实体。

**识别流程：**

1. 将文本分成单个字："王明在广州" → ['王', '明', '在', '广', '州']
2. 为每个字提取特征
3. 用CRF模型预测每个字的标签：['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']
4. 根据标签合并成实体："王明"(PER), "广州"(LOC)

让我们逐步实现这个流程。

### 5.13 测试新句子 - 第2步：文本预处理

In [ ]:
# 步骤1：将文本转换为CRF需要的格式
def text_to_sent(text):
    """
    将文本转换为句子格式
    
    输入: "王明在广州"
    输出: [('王', 'O'), ('明', 'O'), ('在', 'O'), ('广', 'O'), ('州', 'O')]
    
    注：这里的'O'是占位符，实际标签由CRF预测
    """
    chars = list(text)  # 分成单个字
    sent = [(char, 'O') for char in chars]  # 每个字配一个占位符标签
    return sent

# 测试
test_text = "王明在广州"
sent = text_to_sent(test_text)
print(f"原文本: {test_text}")
print(f"转换后: {sent}")

### 5.14 测试新句子 - 第3步：预测标签

In [ ]:
# 步骤2：使用CRF预测标签
def predict_labels(text, crf_model):
    """
    预测文本的标签序列
    
    返回: 字符列表, 标签列表
    """
    # 转换为句子格式
    sent = text_to_sent(text)
    
    # 提取特征
    features = sent2features(sent)
    
    # 预测标签
    labels = crf_model.predict([features])[0]
    
    # 获取字符列表
    chars = [char for char, _ in sent]
    
    return chars, labels

# 测试
chars, labels = predict_labels(test_text, crf)
print(f"文本: {test_text}\n")
print("字符和标签：")
for char, label in zip(chars, labels):
    print(f"  {char} → {label}")

### 5.15 测试新句子 - 第4步：合并实体

In [ ]:
# 步骤3：根据标签合并连续的实体
def merge_entities(chars, labels):
    """
    将连续的B-I标签合并成实体
    
    输入: 
        chars = ['王', '明', '在', '广', '州']
        labels = ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']
    
    输出:
        [('王明', 'PER'), ('广州', 'LOC')]
    """
    entities = []  # 存储识别出的实体
    current_entity = []  # 当前正在构建的实体
    current_type = None  # 当前实体的类型
    
    for char, label in zip(chars, labels):
        if label.startswith('B-'):  # 新实体开始
            # 保存之前的实体（如果有）
            if current_entity:
                entity_text = ''.join(current_entity)
                entities.append((entity_text, current_type))
            
            # 开始新实体
            current_entity = [char]
            current_type = label[2:]  # 去掉'B-'前缀，得到类型
            
        elif label.startswith('I-') and current_type == label[2:]:
            # 实体继续（类型必须匹配）
            current_entity.append(char)
            
        else:  # 遇到'O'或不匹配的标签
            # 保存当前实体（如果有）
            if current_entity:
                entity_text = ''.join(current_entity)
                entities.append((entity_text, current_type))
                current_entity = []
                current_type = None
    
    # 处理最后一个实体
    if current_entity:
        entity_text = ''.join(current_entity)
        entities.append((entity_text, current_type))
    
    return entities

# 测试
entities = merge_entities(chars, labels)
print(f"识别出的实体：")
for entity, etype in entities:
    print(f"  [{entity}] ({etype})")

**合并逻辑说明：**

- 遇到 `B-XXX`：开始一个新实体
- 遇到 `I-XXX`（类型匹配）：继续当前实体
- 遇到 `O` 或其他：结束当前实体

例如：`['B-PER', 'I-PER', 'O', 'B-LOC']` → 两个实体：第1-2个字是PER，第4个字是LOC的开始

### 5.16 测试新句子 - 第5步：完整的预测函数

In [ ]:
# 完整版：整合所有步骤
def predict_ner(text, crf_model):
    """
    对新文本进行命名实体识别
    
    参数:
        text: 输入文本
        crf_model: 训练好的CRF模型
    
    返回:
        chars: 字符列表
        labels: 标签列表
        entities: 识别出的实体列表 [(实体文本, 类型), ...]
    """
    # 步骤1+2：预测标签
    chars, labels = predict_labels(text, crf_model)
    
    # 步骤3：合并实体
    entities = merge_entities(chars, labels)
    
    return chars, labels, entities

# 测试
chars, labels, entities = predict_ner(test_text, crf)
print(f"文本: {test_text}\n")
print("标注结果: ", end="")
for char, label in zip(chars, labels):
    print(f"{char}/{label} ", end="")
print("\n\n识别出的实体:")
for entity, etype in entities:
    print(f"  [{entity}]({etype})")

### 5.17 测试新句子 - 第6步：批量测试

In [ ]:
# 准备多个测试句子
test_texts = [
    "王明在广州工作",
    "华为在深圳",
    "李四在北京创业",
    "张三去上海出差",
]

print("新句子实体识别测试")
print("="*80)

for text in test_texts:
    chars, labels, entities = predict_ner(text, crf)
    
    print(f"\n文本: {text}")
    
    # 显示标注
    print("标注: ", end="")
    for char, label in zip(chars, labels):
        print(f"{char}/{label} ", end="")
    
    # 显示识别的实体
    print("\n识别出的实体:")
    if entities:
        for entity, etype in entities:
            # 实体类型的中文名称
            type_names = {'PER': '人名', 'LOC': '地名', 'ORG': '机构名'}
            type_cn = type_names.get(etype, etype)
            print(f"  [{entity}]({type_cn})")
    else:
        print("  无")
    
    print("-"*80)

**测试结果分析：**

观察测试结果，你会发现：

**1. 能识别的情况：**
- 类似训练数据的人名、地名（如"王明"、"广州"）
- 常见的双字人名和地名

**2. 可能识别错误的情况：**
- 训练数据中没见过的机构名（如"华为"可能被识别为人名）
- 特殊的实体（如四字的"字节跳动"）

**3. 为什么会错？**
- 训练数据太少（只有8个句子）
- 特征不够丰富（没有使用词典等高级特征）

**4. 如何改进？**
- 增加训练数据（几百到几千个句子）
- 添加更多特征（词典、词性等）
- 使用深度学习方法（下节课内容）

## 6. 课程总结

### 6.1 本节课我们学习了：

**1. 序列标注问题**
- 将NER看作序列标注任务
- 给每个字或词分配一个标签

**2. BIO标注方案**
- B：实体开始
- I：实体中间
- O：非实体
- 可以明确标识实体边界

**3. 条件随机场（CRF）**
- 考虑上下文信息
- 考虑标签之间的转移规律
- 保证标签序列的合理性

**4. 特征工程**
- 字符特征
- 上下文特征
- 词典特征
- 特征对模型效果影响很大

**5. 实践应用**
- 使用sklearn-crfsuite库
- 训练和评估CRF模型
- 对新文本进行实体识别

### 6.2 关键要点

**相比规则方法的优势：**
- ✅ 可以识别未登录词（新词）
- ✅ 可以利用上下文消歧
- ✅ 不需要大量人工编写规则
- ✅ 可以从数据中自动学习规律

**局限性：**
- ❌ 需要大量标注数据
- ❌ 特征工程需要领域知识
- ❌ 对于复杂场景效果有限

**实践建议：**
- 标注数据要有代表性
- 特征设计要结合领域知识
- 可以将规则方法作为特征加入CRF
- 持续迭代优化特征和数据

### 6.3 方法对比

| 特点 | 规则方法 | CRF方法 |
|------|---------|--------|
| 数据需求 | 不需要标注数据 | 需要标注数据 |
| 开发难度 | 简单 | 中等 |
| 识别准确率 | 中等 | 较高 |
| 处理新词能力 | 弱 | 中等 |
| 上下文理解 | 无 | 较强 |
| 可解释性 | 强 | 中等 |
| 领域迁移 | 需要新规则 | 需要新数据 |

**选择建议：**
- 数据不足：使用规则方法
- 有标注数据：使用CRF方法
- 追求更高效果：使用深度学习方法
- 实际应用：多种方法结合